In [30]:
import ast
from dataclasses import dataclass
from typing import Any

import sympy as sp
from sympy.printing.latex import LatexPrinter


@dataclass(frozen=True)
class Meta:
    kind: str
    children: tuple[Any, ...]
    func: Any = None
    source: str = ""


class OrderedSympyBuilder(ast.NodeVisitor):
    """
    Builds one real SymPy expression, plus sidecar presentation metadata.

    The metadata preserves how the engineer wrote:
    - Add order
    - Subtract order
    - Multiply order
    - Divide form
    - Power form
    - Function argument order

    SymPy is still the mathematics engine.
    """

    def __init__(self, source: str, namespace=None):
        self.source = source
        self.namespace = dict(namespace or {})
        self.meta = {}

    def remember(self, expr, meta):
        self.meta[id(expr)] = meta
        return expr

    def build(self):
        tree = ast.parse(self.source, mode="eval")
        return self.visit(tree.body)

    def src(self, node):
        return ast.get_source_segment(self.source, node) or ""

    def visit_Name(self, node):
        constants = {
            "pi": sp.pi,
            "E": sp.E,
            "oo": sp.oo,
            "True": sp.true,
            "False": sp.false,
        }

        if node.id in self.namespace:
            return self.namespace[node.id]

        if node.id in constants:
            return constants[node.id]

        return sp.Symbol(node.id)

    def visit_Constant(self, node):
        if isinstance(node.value, bool):
            return sp.true if node.value else sp.false

        if isinstance(node.value, int):
            return sp.Integer(node.value)

        if isinstance(node.value, float):
            return sp.Float(str(node.value))

        raise ValueError(f"Unsupported constant: {node.value!r}")

    def visit_Tuple(self, node):
        return tuple(self.visit(e) for e in node.elts)

    def visit_UnaryOp(self, node):
        x = self.visit(node.operand)

        if isinstance(node.op, ast.USub):
            expr = sp.Mul(sp.Integer(-1), x, evaluate=False)
            return self.remember(expr, Meta("neg", (x,), source=self.src(node)))

        if isinstance(node.op, ast.UAdd):
            return x

        raise ValueError(f"Unsupported unary operator: {ast.dump(node.op)}")

    def visit_BinOp(self, node):
        left = self.visit(node.left)
        right = self.visit(node.right)

        if isinstance(node.op, ast.Add):
            expr = sp.Add(left, right, evaluate=False)
            kind = "add"

        elif isinstance(node.op, ast.Sub):
            expr = sp.Add(
                left,
                sp.Mul(-1, right, evaluate=False),
                evaluate=False,
            )
            kind = "sub"

        elif isinstance(node.op, ast.Mult):
            expr = sp.Mul(left, right, evaluate=False)
            kind = "mul"

        elif isinstance(node.op, ast.Div):
            expr = sp.Mul(
                left,
                sp.Pow(right, -1, evaluate=False),
                evaluate=False,
            )
            kind = "div"

        elif isinstance(node.op, ast.Pow):
            expr = sp.Pow(left, right, evaluate=False)
            kind = "pow"

        else:
            raise ValueError(f"Unsupported operator: {ast.dump(node.op)}")

        return self.remember(
            expr,
            Meta(kind, (left, right), source=self.src(node)),
        )

    def visit_Call(self, node):
        func = self.resolve_function(node.func)
        args = tuple(self.visit(a) for a in node.args)

        try:
            expr = func(*args, evaluate=False)
        except TypeError:
            expr = func(*args)

        return self.remember(
            expr,
            Meta("call", args, func=func, source=self.src(node)),
        )

    def resolve_function(self, node):
        if isinstance(node, ast.Name):
            name = node.id

            if name in self.namespace:
                return self.namespace[name]

            obj = getattr(sp, name, None)

            if callable(obj):
                return obj

            # Unknown functions are still valid SymPy undefined functions.
            return sp.Function(name)

        if isinstance(node, ast.Attribute):
            # Allows sp.sin(x), sympy.sin(x), etc.
            name = node.attr
            obj = getattr(sp, name, None)

            if callable(obj):
                return obj

        raise ValueError(f"Unsupported function reference: {ast.dump(node)}")

    def generic_visit(self, node):
        raise ValueError(f"Unsupported syntax: {ast.dump(node)}")


class OrderedLatexPrinter(LatexPrinter):
    """
    Custom printer that only controls presentation order.

    It does NOT hardcode sin, cos, sqrt, log, Integral, Derivative, etc.

    Function rendering is delegated to SymPy.
    """

    def __init__(self, meta=None, substitutions=None, **settings):
        self.meta = meta or {}
        self.substitutions = substitutions or {}
        self._plain_printer = LatexPrinter(settings)
        super().__init__(settings)

    def m(self, expr):
        return self.meta.get(id(expr))

    def _print(self, expr, **kwargs):
        meta = self.m(expr)

        if meta and meta.kind == "call" and getattr(expr, "is_Function", False):
            rendered = self._try_print_function_call(expr, meta)

            if rendered is not None:
                return rendered

        return super()._print(expr, **kwargs)

    def _try_print_function_call(self, expr, meta):
        """
        Generic function renderer.

        Example:
            original: Max(b, a)

        SymPy internally may store:
            Max(a, b)

        This method creates:
            Max(HCARG0, HCARG1)

        Lets SymPy render the shell:
            \\max(HCARG0, HCARG1)

        Then replaces:
            HCARG0 -> b
            HCARG1 -> a

        So no function-specific LaTeX mapping is needed.
        """

        if any(isinstance(a, tuple) for a in meta.children):
            return None

        placeholders = [
            sp.Symbol(f"HCARG{i}")
            for i in range(len(meta.children))
        ]

        try:
            shell_expr = meta.func(*placeholders, evaluate=False)
        except TypeError:
            try:
                shell_expr = meta.func(*placeholders)
            except Exception:
                return None
        except Exception:
            return None

        shell = self._plain_printer.doprint(shell_expr)

        for placeholder, actual in zip(placeholders, meta.children):
            shell = shell.replace(
                self._plain_printer.doprint(placeholder),
                self._print_arg(actual),
            )

        return shell

    def _print_arg(self, arg):
        if isinstance(arg, tuple):
            return (
                r"\left("
                + ", ".join(self._print_arg(a) for a in arg)
                + r"\right)"
            )

        return self._print(arg)

    def _print_Symbol(self, expr):
        if expr in self.substitutions:
            return self._print(self.substitutions[expr])

        return super()._print_Symbol(expr)

    def _paren(self, s):
        return r"\left(" + s + r"\right)"

    def _symbol_is_substituted(self, expr):
        return isinstance(expr, sp.Symbol) and expr in self.substitutions

    def _child(self, expr, parent=None, side=None):
        s = self._print_arg(expr)

        if self._symbol_is_substituted(expr):
            if parent == "mul" and side != "first":
                return self._paren(s)

            if parent == "pow" and side == "base":
                return self._paren(s)

        if isinstance(expr, sp.Add) and parent in {"mul", "pow", "sub"}:
            return self._paren(s)

        return s

    def _print_Add(self, expr, order=None):
        meta = self.m(expr)

        if meta and meta.kind in {"add", "sub"}:
            left, right = meta.children
            op = " + " if meta.kind == "add" else " - "

            return (
                self._child(left, meta.kind, "left")
                + op
                + self._child(right, meta.kind, "right")
            )

        # Fallback: use existing args order, not SymPy sorted output.
        parts = []

        for i, arg in enumerate(expr.args):
            if arg.could_extract_minus_sign():
                core = -arg
                sign = " - " if i else "-"
            else:
                core = arg
                sign = " + " if i else ""

            parts.append(sign + self._print_arg(core))

        return "".join(parts)

    def _print_Mul(self, expr):
        meta = self.m(expr)

        if meta and meta.kind == "mul":
            left, right = meta.children

            return (
                self._child(left, "mul", "first")
                + self._child(right, "mul", "right")
            )

        if meta and meta.kind == "div":
            left, right = meta.children

            return (
                r"\frac{"
                + self._print_arg(left)
                + "}{"
                + self._print_arg(right)
                + "}"
            )

        if meta and meta.kind == "neg":
            (child,) = meta.children
            return "-" + self._child(child, "neg", "right")

        return super()._print_Mul(expr)

    def _print_Pow(self, expr):
        meta = self.m(expr)

        if meta and meta.kind == "pow":
            base, exponent = meta.children

            return (
                self._child(base, "pow", "base")
                + "^{"
                + self._print_arg(exponent)
                + "}"
            )

        return super()._print_Pow(expr)


def exact_value(v):
    if isinstance(v, float):
        return sp.Rational(str(v))

    return sp.sympify(v)


def display_value(v):
    return sp.sympify(str(v))


def clean_number(expr, sig=12):
    n = sp.N(expr, sig)

    try:
        f = float(n)

        if abs(f - round(f)) < 1e-10:
            return str(int(round(f)))

        return f"{f:.{sig}g}"

    except Exception:
        return sp.latex(n)


class CalcExpression:
    """
    Single-source calculation object.

    Engineer writes the equation once.

    This object then owns:
    - source text
    - real SymPy expression
    - ordered presentation metadata
    - numerical values
    """

    def __init__(self, lhs, expression, namespace=None, **values):
        self.lhs = lhs
        self.source = expression

        builder = OrderedSympyBuilder(expression, namespace=namespace)
        self.sympy = builder.build()
        self.meta = builder.meta

        self.calc_subs = {
            sp.Symbol(k): exact_value(v)
            for k, v in values.items()
        }

        self.display_subs = {
            sp.Symbol(k): display_value(v)
            for k, v in values.items()
        }

    def latex(self, substituted=False):
        printer = OrderedLatexPrinter(
            self.meta,
            substitutions=self.display_subs if substituted else {},
            mul_symbol="",
        )

        return printer.doprint(self.sympy)

    def result(self):
        return self.sympy.subs(self.calc_subs).doit()

    def report(self):
        return "\n".join(
            [
                rf"{self.lhs} = {self.latex(False)}",
                "=",
                rf"{self.lhs} = {self.latex(True)}",
                "=",
                rf"{self.lhs} = {clean_number(self.result())}",
            ]
        )
    def report_latex(self):
        return rf"""
    \begin{{aligned}}
    {self.lhs} &= {self.latex(False)} \\[6pt]
    &= {self.latex(True)} \\[6pt]
    &= {clean_number(self.result())}
    \end{{aligned}}
    """

In [43]:
M = CalcExpression(
    "M",
    "pi*e + w*L**2/8",
    P=500,
    e=0.2,
    w=10,
    L=5,
)

print(M.report_latex())


    \begin{aligned}
    M &= \pie + \frac{wL^{2}}{8} \\[6pt]
    &= \pi\left(0.2\right) + \frac{10\left(5\right)^{2}}{8} \\[6pt]
    &= 31.8783185307
    \end{aligned}
    


In [33]:
R = CalcExpression(
    "R",
    "x**2 + 3*x + y",
    x=5,
    y=2,
)

print(R.report())

R = x^{2} + 3x + y
=
R = \left(5\right)^{2} + 3\left(5\right) + 2
=
R = 42


In [48]:
display(Math(R.report_latex()))

<IPython.core.display.Math object>

In [34]:
from IPython.display import display, Math
Z = CalcExpression(
    "Z",
    "sin(x**2 + 3*x + y) + log(a*b) + sqrt(q) + Max(b, a)",
    x=5,
    y=2,
    a=10,
    b=100,
    q=16,
)

display(Math((Z.report_latex())))

<IPython.core.display.Math object>

In [35]:
Q = CalcExpression(
    "Q",
    "Integral(sin(t), (t, 0, pi)) + Subs(Derivative(t**2 + 3*t, t), t, x)",
    x=5,
)

print(Q.report())

Q = \int\limits_{0}^{\pi} \sin{\left(t \right)}\, dt + \left. \frac{d}{d t} \left(t^{2} + 3t\right) \right|_{\substack{ t=x }}
=
Q = \int\limits_{0}^{\pi} \sin{\left(t \right)}\, dt + \left. \frac{d}{d t} \left(t^{2} + 3t\right) \right|_{\substack{ t=5 }}
=
Q = 15


In [36]:
display(Math((Q.report_latex())))

<IPython.core.display.Math object>

In [37]:
import sympy as sp
from IPython.display import display, Math

# ----------------------------------------------------------
# Structural engineering calculation helpers
# ----------------------------------------------------------

def area_of_bar(d):
    """
    Area of one circular reinforcement bar.
    A_s = pi*d^2/4
    """
    return CalcExpression(
        "A_s",
        "pi*d**2/4",
        d=d
    )


def total_area_of_bars(n, d):
    """
    Total area of n reinforcement bars.
    A_s = n*pi*d^2/4
    """
    return CalcExpression(
        "A_s",
        "n*pi*d**2/4",
        n=n,
        d=d
    )


def required_steel_area(M_Ed, z, f_yd):
    """
    Required tensile reinforcement area.
    A_s,req = M_Ed / (z*f_yd)
    """
    return CalcExpression(
        "A_{s,req}",
        "M_Ed/(z*f_yd)",
        M_Ed=M_Ed,
        z=z,
        f_yd=f_yd
    )


def lever_arm(d, x):
    """
    Lever arm.
    z = d - x/3
    """
    return CalcExpression(
        "z",
        "d - x/3",
        d=d,
        x=x
    )


def concrete_area_rectangular(b, h):
    """
    Rectangular concrete area.
    A_c = b*h
    """
    return CalcExpression(
        "A_c",
        "b*h",
        b=b,
        h=h
    )


def second_moment_rectangular(b, h):
    """
    Second moment of area of rectangle about strong axis.
    I = b*h^3/12
    """
    return CalcExpression(
        "I",
        "b*h**3/12",
        b=b,
        h=h
    )


def section_modulus_rectangular(b, h):
    """
    Elastic section modulus of rectangle.
    W = b*h^2/6
    """
    return CalcExpression(
        "W",
        "b*h**2/6",
        b=b,
        h=h
    )


def bending_stress(M, W):
    """
    Bending stress.
    sigma = M/W
    """
    return CalcExpression(
        r"\sigma",
        "M/W",
        M=M,
        W=W
    )


def axial_stress(N, A):
    """
    Axial stress.
    sigma = N/A
    """
    return CalcExpression(
        r"\sigma",
        "N/A",
        N=N,
        A=A
    )


def combined_stress(N, A, M, W):
    """
    Combined axial + bending stress.
    sigma = N/A + M/W
    """
    return CalcExpression(
        r"\sigma",
        "N/A + M/W",
        N=N,
        A=A,
        M=M,
        W=W
    )

In [38]:
As_bar = area_of_bar(d=16)
display(Math(As_bar.report_latex()))

As_total = total_area_of_bars(n=4, d=16)
display(Math(As_total.report_latex()))

I_rect = second_moment_rectangular(b=300, h=500)
display(Math(I_rect.report_latex()))

sigma = combined_stress(
    N=100000,
    A=150000,
    M=50e6,
    W=12.5e6
)
display(Math(sigma.report_latex()))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [40]:
calculations = [
    area_of_bar(d=16),
    total_area_of_bars(n=4, d=16),
    concrete_area_rectangular(b=300, h=500),
    second_moment_rectangular(b=300, h=500),
    section_modulus_rectangular(b=300, h=500),
    combined_stress(N=100000, A=150000, M=50e6, W=12.5e6),
]

for calc in calculations:
    display(Math(calc.report_latex()))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [47]:
A=CalcExpression(
        "A_s",
        "pi * d**2/4",
        d=10
    )
print((A.report()))

A_s = \frac{\pid^{2}}{4}
=
A_s = \frac{\pi\left(10\right)^{2}}{4}
=
A_s = 78.5398163397


In [46]:
print(sp.latex(sp.pi))

\pi
